In [ ]:
## Testing and Evaluation Pipeline , import required files and libraries
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
project_path = Path("/content/drive/MyDrive/ISL_GAN_FewShot_Project")

landmark_dir = project_path / "data" / "landmarks"
augmented_dir = project_path / "data" / "augmented"

baseline_model_candidates = [
    project_path / "models" / "baseline" / "baseline_mlp_best.pth",
    project_path / "models" / "baseline_mlp_best.pth"
]

final_model_candidates = [
    project_path / "models" / "fewshot" / "final_augmented_mlp_best.pth",
    project_path / "models" / "final_augmented_mlp_best.pth"
]

report_dir = project_path / "results"
report_dir.mkdir(parents=True, exist_ok=True)

with open(landmark_dir / "class_mapping.json", "r") as f:
    mapping = json.load(f)

class_to_idx = mapping["class_to_idx"]
idx_to_class = {int(k): v for k, v in mapping["idx_to_class"].items()}
num_classes = len(class_to_idx)

print("Project ready.")
print("Number of classes:", num_classes)

In [ ]:
X_test = np.load(landmark_dir / "X_test.npy")
y_test = np.load(landmark_dir / "y_test.npy")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_dataset = LandmarkDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def find_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these model paths exist: {candidates}")

def evaluate_model(model, loader, device):
    model.eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)

            all_true.extend(y_batch.numpy())
            all_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true, all_pred, average="macro", zero_division=0
    )
    return acc, precision, recall, f1, all_true, all_pred